<a href="https://colab.research.google.com/github/ragiokay/AI_transcribe/blob/main/AI_transcribe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1：環境設定與基礎配置

In [ ]:
!pip install faster-whisper google-generativeai python-dotenv

import os
import time
from google.colab import drive
import google.generativeai as genai
from google.colab import userdata
from faster_whisper import WhisperModel

print("[開始] 正在掛載 Google Drive...")
drive.mount('/content/drive')

# 設定資料夾與檔名
DRIVE_DIR = "/content/drive/MyDrive/AI_transcribe"
os.makedirs(DRIVE_DIR, exist_ok=True)

BASE_NAME = "NLP_Group5_Meeting"
AUDIO_FILE_PATH = os.path.join(DRIVE_DIR, f"{BASE_NAME}.m4a")

# 定義各階段的檔案路徑
PATH_1_RAW = os.path.join(DRIVE_DIR, f"{BASE_NAME}_1_原始逐字稿.txt")
PATH_2_REPAIR = os.path.join(DRIVE_DIR, f"{BASE_NAME}_2_音節修復保留冗詞版.md")
PATH_4_CLEAN = os.path.join(DRIVE_DIR, f"{BASE_NAME}_4_刪除冗詞版.md")
PATH_6_CHRONO_SUM = os.path.join(DRIVE_DIR, f"{BASE_NAME}_6_時間序局部摘要.md")
PATH_8_FINAL_SUM = os.path.join(DRIVE_DIR, f"{BASE_NAME}_8_最終重構摘要.md")

# 安全讀取 API Key (請確保你已在 Colab Secrets 設定 GEMINI_API_KEY)
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

# 使用速度與長文本能力兼具的模型
LLM_MODEL_NAME = "gemini-3.5-flash"

Cell 2：[Step 1] Whisper 強效轉錄
(加上了專有名詞提示與 VAD 過濾器，產出最精準的原始稿)

In [ ]:
if os.path.exists(PATH_1_RAW):
    print(f"✅ 發現 [1_原始逐字稿]，跳過此步驟。\n路徑: {PATH_1_RAW}")
else:
    print("🚀 開始進行 Whisper 強效轉錄...")
    if not os.path.exists(AUDIO_FILE_PATH):
        raise FileNotFoundError(f"找不到音檔: {AUDIO_FILE_PATH}")

    # 加入 VAD 過濾器消除無聲片段，避免模型幻覺
    model = WhisperModel("large-v3", device="cuda", compute_type="float16")

    # 專屬 Initial Prompt：幫助模型在遇到模糊音節時，優先匹配這些專有名詞
    custom_prompt = "這是一場資工所自然語言處理(NLP)專案討論，包含：Ground Truth、Baseline、Few-shot、LLM、Prompt、Cofacts、Fake、True 等專有名詞。"

    segments, info = model.transcribe(
        AUDIO_FILE_PATH,
        beam_size=5,
        language="zh",
        vad_filter=True,
        initial_prompt=custom_prompt
    )

    raw_text = ""
    for segment in segments:
        line = f"[{segment.start:.1f}s - {segment.end:.1f}s] {segment.text}\n"
        print(line.strip())
        raw_text += line

    with open(PATH_1_RAW, "w", encoding="utf-8") as f:
        f.write(raw_text)
    print(f"\n💾 [1_原始逐字稿] 已儲存至: {PATH_1_RAW}")

Cell 3：[Step 2] 沉浸式音節修復與語者標記 (禁止刪減)
(跑完這格後，你可以打開檔案進行 [Step 3] 人工聽音修復)

In [ ]:
if os.path.exists(PATH_2_REPAIR):
    print(f"✅ 發現 [2_音節修復版]，跳過此步驟。\n請確保你已經人工檢查並修改過這個檔案 (Step 3)！")
else:
    print("🧠 開始進行沉浸式音節修復 (嚴格保留冗詞)...")

    with open(PATH_1_RAW, "r", encoding="utf-8") as f:
        raw_text = f.read()

    system_prompt = """
    你是一位極度嚴謹的語音辨識修復員。這是一場資工所關於「NLP 期末專案」的討論錄音稿，包含時間軸。

    【任務最高指導原則】：絕對禁止刪除任何字詞！
    1. 即使是「呃」、「然後」、「那個」、「就是說」等冗言贅字，或是講者結巴、重複的話，也【必須 100% 保留】。
    2. 你的唯一任務是「音節修復」：根據上下文邏輯，將 ASR 聽錯的同音異義詞或專有名詞修正（例如將「光速」修復為「Ground Truth」，將「恩囉比」修復為「NLP」）。
    3. 語者辨識與時間軸：請根據對話的語氣切換，標記 [講者A]、[講者B] 等。將同一位講者連續發言的時間軸合併，並保留時間標記。

    格式範例：
    [0.0s - 15.5s] [講者A]：呃... 我們會先講一下我們的實驗流程。我們是用那個 Cofacts 這個資料集，然後...
    [15.5s - 22.0s] [講者B]：那你們跟情緒的關係是怎樣？
    """

    model = genai.GenerativeModel(model_name=LLM_MODEL_NAME, system_instruction=system_prompt)
    response = model.generate_content(raw_text, generation_config=genai.GenerationConfig(temperature=0.1))

    with open(PATH_2_REPAIR, "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"✨ [2_音節修復版] 已儲存！\n⚠️ 請前往 Google Drive 打開 {PATH_2_REPAIR} 進行 [Step 3] 人工聽音修復，確認無誤後再執行下一個 Cell。")

Cell 4：[Step 4 & 5] 安全刪除冗詞
(讀取你人工修復後的檔案，清洗口語贅字但不做摘要)

In [ ]:
if os.path.exists(PATH_4_CLEAN):
    print(f"✅ 發現 [4_刪除冗詞版]，跳過此步驟。")
else:
    print("🧠 讀取人工修復版，開始進行口語冗詞清洗...")

    if not os.path.exists(PATH_2_REPAIR):
        raise FileNotFoundError("請先完成 Step 2 與 Step 3！")

    with open(PATH_2_REPAIR, "r", encoding="utf-8") as f:
        repaired_text = f.read()

    system_prompt = """
    你是一位專業編輯。我會給你一份已標記語者與時間軸的逐字稿。
    任務：清洗無意義的口語贅詞（如「呃」、「然後」、「那個」、「對」），並修正破碎的語法使其通順。
    嚴格限制：【絕對不可摘要】。任何助教的提問、學生的細節回答、數據討論，都必須完整保留，只能刪除純粹的發語詞與結巴。保留時間軸與講者標記。
    """

    model = genai.GenerativeModel(model_name=LLM_MODEL_NAME, system_instruction=system_prompt)
    response = model.generate_content(repaired_text, generation_config=genai.GenerationConfig(temperature=0.1))

    with open(PATH_4_CLEAN, "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"✨ [4_刪除冗詞版] 已儲存至: {PATH_4_CLEAN}")

Cell 5：[Step 6] 依時間序局部摘要
(跑完這格後，你可以打開檔案進行 [Step 7] 人工補充摘要)

In [ ]:
if os.path.exists(PATH_6_CHRONO_SUM):
    print(f"✅ 發現 [6_時間序局部摘要]，跳過此步驟。\n請確保你已經人工檢查並補充過這個檔案 (Step 7)！")
else:
    print("🧠 開始依照時間軸進行局部重點摘要...")

    with open(PATH_4_CLEAN, "r", encoding="utf-8") as f:
        clean_text = f.read()

    system_prompt = """
    你是一位會議紀錄助理。請根據提供的逐字稿，撰寫一份「依時間序排列的局部摘要」。
    任務：
    1. 順著時間軸，每當對話話題轉換時，切分出一個段落。
    2. 標註該段落的時間區間（例如 [0.0s - 120.0s]）。
    3. 以條列式寫出該區間內討論的具體重點、提出的質疑或建議。切勿遺漏任何關鍵數字（如 p-value, 題數）與技術細節。
    """

    model = genai.GenerativeModel(model_name=LLM_MODEL_NAME, system_instruction=system_prompt)
    response = model.generate_content(clean_text, generation_config=genai.GenerationConfig(temperature=0.2))

    with open(PATH_6_CHRONO_SUM, "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"✨ [6_時間序局部摘要] 已儲存！\n⚠️ 請前往 Google Drive 打開 {PATH_6_CHRONO_SUM} 進行 [Step 7] 人工重點補充，確認無誤後再執行最後一個 Cell。")

Cell 6：[Step 8] 解析重構綜合摘要
(讀取你確認過的時間序摘要，輸出最終結構化報告)

In [ ]:
if os.path.exists(PATH_8_FINAL_SUM):
    print(f"✅ 發現 [8_最終重構摘要]，全部流程已完成！檔案位置: {PATH_8_FINAL_SUM}")
else:
    print("🧠 讀取確認後的局部摘要，開始重構最終綜合會議紀錄...")

    if not os.path.exists(PATH_6_CHRONO_SUM):
        raise FileNotFoundError("請先完成 Step 6 與 Step 7！")

    with open(PATH_6_CHRONO_SUM, "r", encoding="utf-8") as f:
        chrono_text = f.read()

    system_prompt = """
    你是一位高級技術架構師與專案經理。我會提供一份依時間序排列的會議重點摘要。
    請幫我打破時間軸的限制，將這些內容「解析重構」為一份專業、結構化的最終會議紀錄。

    輸出格式必須包含以下 Markdown 標題（若有相關內容）：
    ## 1. 專案現狀與實驗進度
    ## 2. 核心討論與數據發現
    ## 3. 助教回饋與技術質疑
    ## 4. 下一步行動 (Action Items)

    請確保所有專有名詞精準，並維持學術嚴謹的語氣。
    """

    model = genai.GenerativeModel(model_name=LLM_MODEL_NAME, system_instruction=system_prompt)
    response = model.generate_content(chrono_text, generation_config=genai.GenerationConfig(temperature=0.2))

    with open(PATH_8_FINAL_SUM, "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"🎉 任務圓滿結束！最終重構報告已安全儲存至: {PATH_8_FINAL_SUM}")